# 02. Benchmark: $\beta$-TCVAE with Minibatch Weighted Sampling (MWS)

This notebook implements, trains, and evaluates **$\beta$-TCVAE** ($\beta = 6.0$) using the Minibatch Weighted Sampling (MWS) Total Correlation estimator (Chen et al., 2018).

---

### Theoretical Grounding: The KL Divergence Decomposition
Chen et al. (2018, Section 3, Eq. 2) proved that the average KL divergence term in the VAE ELBO decomposes into three information-theoretic quantities:
$$\mathbb{E}_{p(n)} [ D_{\text{KL}}(q(z|n) \parallel p(z)) ] = \underbrace{I_q(x; z)}_{\text{i. Index-Code MI}} + \underbrace{\text{TC}(z)}_{\text{ii. Total Correlation}} + \underbrace{\sum_{j=1}^d D_{\text{KL}}(q(z_j) \parallel p(z_j))}_{\text{iii. Dimension-wise KL}}$$

- **Index-Code MI $I_q(x; z)$**: Quantifies the information between input $x$ and code $z$. Penalizing this term (as $\beta$-VAE does when $\beta > 1$) degrades reconstruction quality.
- **Total Correlation $\text{TC}(z) = D_{\text{KL}}(q(z) \parallel \prod_j q(z_j))$**: Directly measures multi-variable mutual dependence across latent coordinates. Penalizing TC forces the aggregated posterior to factorize without constricting $I_q(x; z)$.
- **Dimension-wise KL**: Enforces individual coordinate marginals to stay close to standard priors $\mathcal{N}(0, 1)$.

The generalized $\beta$-TCVAE objective (Chen et al. 2018, Eq. 4) sets $\alpha = 1, \gamma = 1$ and modifies $\beta > 1$:
$$\mathcal{L}_{\beta\text{-TC}} = \mathbb{E}_{q(z|x)}[\log p(x|z)] - I_q(x; z) - \beta \text{TC}(z) - \sum_{j=1}^d D_{\text{KL}}(q(z_j) \parallel p(z_j))$$
$$= \mathcal{L}_{\text{ELBO}} - (\beta - 1) \text{TC}(z)$$

---

### Minibatch Weighted Sampling (MWS) Estimator
To evaluate Total Correlation without auxiliary discriminator networks, MWS estimates $\log q(z^{(i)})$ over a minibatch $\mathcal{B}$ of size $M$ sampled from a dataset of size $N$:
$$\log q(z^{(i)}) \approx \log \left( \frac{1}{M N} \sum_{k=1}^M q(z^{(i)} | x^{(k)}) \right)$$
$$\log q_j(z_j^{(i)}) \approx \log \left( \frac{1}{M N} \sum_{k=1}^M q(z_j^{(i)} | x^{(k)}) \right)$$
$$\widehat{\text{TC}}(z) = \frac{1}{M} \sum_{i=1}^M \left[ \log q(z^{(i)}) - \sum_{j=1}^d \log q_j(z_j^{(i)}) \right]$$

---

### Key Capabilities:
- Drop-in ready for Kaggle/Colab with GPU auto-detection.
- Auto-downloading dSprites dataset.
- `FAST_MODE = True` toggle for 5-8 minute training on 50,000 samples.
- Complete MWS TC estimator computed via PyTorch tensor broadcasting.
- Full quantitative benchmark evaluation (MIG, DCI, FactorVAE Score, Reconstruction MSE).
- Saves `results_betatcvae.csv` and `traversals_betatcvae.png`.



In [ ]:
import os
import sys
import time
import math
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor

# Auto-detect device: CUDA -> MPS -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"[Device] Using NVIDIA CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("[Device] Using Apple Silicon MPS Acceleration")
else:
    device = torch.device("cpu")
    print("[Device] Using CPU")

# Set random seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
print(f"[Config] Random seed set to {SEED}")



In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
# FAST_MODE = True: trains on 50,000 samples for 15 epochs (~5-8 mins on T4 GPU)
# FAST_MODE = False: trains on all 737,280 samples for 25 epochs
FAST_MODE = True

BATCH_SIZE = 256
EPOCHS = 15 if FAST_MODE else 25
NUM_TRAIN_SAMPLES = 50000 if FAST_MODE else 727280
NUM_EVAL_SAMPLES = 5000 if FAST_MODE else 10000
LATENT_DIM = 10
LEARNING_RATE = 5e-4
BETA = 6.0  # Standard optimal beta for beta-TCVAE (Chen et al. 2018)

print("Beta-TCVAE Experiment Settings:")
print(f"  FAST_MODE:         {FAST_MODE}")
print(f"  Beta (TC Weight):  {BETA}")
print(f"  Batch Size:        {BATCH_SIZE}")
print(f"  Epochs:            {EPOCHS}")
print(f"  Training Samples:  {NUM_TRAIN_SAMPLES}")
print(f"  Evaluation Samples:{NUM_EVAL_SAMPLES}")
print(f"  Latent Dimensions: {LATENT_DIM}")
print(f"  Learning Rate:     {LEARNING_RATE}")



In [ ]:
def get_dsprites_filepath():
    filename = "dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz"
    search_paths = [
        os.path.join(".", filename),
        os.path.join(".", "data", filename),
        os.path.join("..", filename),
        os.path.join("..", "data", filename),
        os.path.join("/kaggle/input/dsprites-dataset", filename),
        os.path.join("/kaggle/input/dsprites", filename),
        os.path.join("/content", filename),
        os.path.join("/content/data", filename)
    ]
    for p in search_paths:
        if os.path.exists(p):
            return p
    return None

def download_dsprites(target_dir="./data"):
    filename = "dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz"
    os.makedirs(target_dir, exist_ok=True)
    target_path = os.path.join(target_dir, filename)
    url = "https://github.com/google-deepmind/dsprites-dataset/raw/master/dsprites_ndarray_co1sh3sc6or40x32y32_64x64.npz"
    print(f"[Dataset] Downloading dSprites dataset from {url}...")
    
    def reporthook(block_num, block_size, total_size):
        if block_num % 1000 == 0:
            downloaded = block_num * block_size
            if total_size > 0:
                percent = downloaded * 100 / total_size
                print(f"  Progress: {percent:.1f}% ({downloaded / (1024*1024):.1f} MB / {total_size / (1024*1024):.1f} MB)")
            else:
                print(f"  Downloaded: {downloaded / (1024*1024):.1f} MB")
                
    urllib.request.urlretrieve(url, target_path, reporthook=reporthook)
    print(f"[Dataset] Download complete: {target_path}")
    return target_path

# Load dSprites
filepath = get_dsprites_filepath()
if filepath is None:
    filepath = download_dsprites()

print(f"[Dataset] Loading data from {filepath}...")
raw_data = np.load(filepath, allow_pickle=True, encoding="latin1")
all_imgs = raw_data["imgs"]  # (737280, 64, 64), uint8
all_latents_classes = raw_data["latents_classes"]  # (737280, 6)
all_latents_values = raw_data["latents_values"]  # (737280, 6)
print(f"[Dataset] Successfully loaded {len(all_imgs)} images of shape {all_imgs.shape[1:]}.")

# Split into train and evaluation subsets
np.random.seed(SEED)
shuffled_idx = np.random.permutation(len(all_imgs))
train_indices = shuffled_idx[:NUM_TRAIN_SAMPLES]
eval_indices = shuffled_idx[NUM_TRAIN_SAMPLES:NUM_TRAIN_SAMPLES + NUM_EVAL_SAMPLES]

train_imgs = all_imgs[train_indices]
train_classes = all_latents_classes[train_indices]
eval_imgs = all_imgs[eval_indices]
eval_classes = all_latents_classes[eval_indices]

class DSpritesDataset(Dataset):
    def __init__(self, imgs, latents_classes):
        self.imgs = torch.from_numpy(imgs).float().unsqueeze(1)
        self.latents_classes = torch.from_numpy(latents_classes).long()

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        return self.imgs[idx], self.latents_classes[idx]

train_dataset = DSpritesDataset(train_imgs, train_classes)
eval_dataset = DSpritesDataset(eval_imgs, eval_classes)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"[Dataset] Train batches: {len(train_loader)} | Eval batches: {len(eval_loader)}")



In [ ]:
class ConvVAE(nn.Module):
    """
    Standard ConvVAE architecture for 64x64 binary images (Chen et al. 2018, Locatello et al. 2019).
    """
    def __init__(self, latent_dim=10):
        super(ConvVAE, self).__init__()
        self.latent_dim = latent_dim
        
        # Encoder: 4 convolutional layers
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),   # (B, 32, 32, 32)
            nn.ReLU(True),
            nn.Conv2d(32, 32, kernel_size=4, stride=2, padding=1),  # (B, 32, 16, 16)
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # (B, 64, 8, 8)
            nn.ReLU(True),
            nn.Conv2d(64, 64, kernel_size=4, stride=2, padding=1),  # (B, 64, 4, 4)
            nn.ReLU(True)
        )
        self.fc_enc = nn.Linear(64 * 4 * 4, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        # Decoder: linear layers + 4 transposed convolutions
        self.fc_dec = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 64 * 4 * 4),
            nn.ReLU(True)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 64, kernel_size=4, stride=2, padding=1), # (B, 64, 8, 8)
            nn.ReLU(True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), # (B, 32, 16, 16)
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 32, kernel_size=4, stride=2, padding=1), # (B, 32, 32, 32)
            nn.ReLU(True),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1)   # (B, 1, 64, 64)
        )

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        h = F.relu(self.fc_enc(h))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), 64, 4, 4)
        recon_logits = self.decoder(h)
        return recon_logits

    def decode_probs(self, z):
        logits = self.decode(z)
        return torch.sigmoid(logits)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_logits = self.decode(z)
        return recon_logits, mu, logvar, z



In [ ]:
def compute_mws_tc(z, mu, logvar, dataset_size):
    """
    Computes Total Correlation via Minibatch Weighted Sampling (MWS) (Chen et al., 2018).
    z: sampled latents (M, d)
    mu: encoder means (M, d)
    logvar: encoder log-variances (M, d)
    dataset_size: total training dataset size N
    """
    M, d = z.size()
    
    # Pairwise log Gaussian density: log q(z_i | x_k)
    # diff: (M, M, d), var: (1, M, d)
    diff = z.unsqueeze(1) - mu.unsqueeze(0)
    var = logvar.exp().unsqueeze(0)
    log_q_zx_per_dim = -0.5 * (math.log(2.0 * math.pi) + logvar.unsqueeze(0) + (diff ** 2) / (var + 1e-8))
    
    # Joint log q(z_i | x_k): sum over dimensions d -> (M, M)
    log_q_zx = log_q_zx_per_dim.sum(dim=2)
    
    # Aggregated log q(z_i): logsumexp over k -> (M,)
    log_qz = torch.logsumexp(log_q_zx, dim=1) - math.log(M * dataset_size)
    
    # Marginal log q(z_ij): logsumexp over k -> (M, d)
    log_qz_j = torch.logsumexp(log_q_zx_per_dim, dim=1) - math.log(M * dataset_size)
    log_qz_prod = log_qz_j.sum(dim=1)  # sum over latent dimensions -> (M,)
    
    # Total Correlation: E_q(z) [ log q(z) - sum_j log q(z_j) ]
    tc = (log_qz - log_qz_prod).mean()
    
    # Mutual Information: E_q(z,x) [ log q(z|x) - log q(z) ]
    log_q_z_given_x = -0.5 * (math.log(2.0 * math.pi) + logvar + ((z - mu) ** 2) / (logvar.exp() + 1e-8)).sum(dim=1)
    mi = (log_q_z_given_x - log_qz).mean()
    
    # Dimension-wise KL: E_q(z) [ sum_j (log q(z_j) - log p(z_j)) ]
    log_pz = -0.5 * (math.log(2.0 * math.pi) + z ** 2).sum(dim=1)
    dw_kl = (log_qz_prod - log_pz).mean()
    
    return tc, mi, dw_kl



In [ ]:
@torch.no_grad()
def compute_reconstruction_mse(model, eval_loader, device):
    """Computes mean squared error between reconstructed and original images."""
    model.eval()
    total_mse = 0.0
    total_samples = 0
    for x, _ in eval_loader:
        x = x.to(device)
        mu, _ = model.encode(x)
        recon = model.decode_probs(mu)
        mse = torch.mean((recon - x) ** 2, dim=[1, 2, 3]).sum().item()
        total_mse += mse
        total_samples += x.size(0)
    return total_mse / total_samples

def compute_mig(latents, factor_classes, num_bins=20):
    """
    Computes Mutual Information Gap (MIG) (Chen et al., 2018).
    latents: np.ndarray (N, d)
    factor_classes: np.ndarray (N, K)
    """
    N, d = latents.shape
    K = factor_classes.shape[1]
    
    # Discretize continuous latents into bins
    discretized = np.zeros_like(latents, dtype=int)
    for j in range(d):
        hist, bin_edges = np.histogram(latents[:, j], bins=num_bins)
        discretized[:, j] = np.clip(np.digitize(latents[:, j], bin_edges[:-1]) - 1, 0, num_bins - 1)
        
    mi_matrix = np.zeros((d, K))
    factor_entropies = np.zeros(K)
    
    for k in range(K):
        fk = factor_classes[:, k]
        unique_f, counts_f = np.unique(fk, return_counts=True)
        if len(unique_f) <= 1:
            factor_entropies[k] = 0.0
            continue
        prob_f = counts_f / N
        factor_entropies[k] = -np.sum(prob_f * np.log(prob_f + 1e-12))
        
        for j in range(d):
            zj = discretized[:, j]
            joint_counts = np.histogram2d(zj, fk, bins=[num_bins, len(unique_f)])[0]
            joint_p = joint_counts / N
            p_z = joint_p.sum(axis=1, keepdims=True)
            p_v = joint_p.sum(axis=0, keepdims=True)
            nz = joint_p > 0
            mi = np.sum(joint_p[nz] * np.log(joint_p[nz] / (p_z @ p_v)[nz] + 1e-12))
            mi_matrix[j, k] = max(0.0, mi)
            
    active_factors = [k for k in range(K) if factor_entropies[k] > 1e-6]
    if len(active_factors) == 0:
        return 0.0
        
    gaps = []
    for k in active_factors:
        sorted_mi = np.sort(mi_matrix[:, k])[::-1]
        gap = (sorted_mi[0] - sorted_mi[1]) / factor_entropies[k]
        gaps.append(gap)
        
    return float(np.mean(gaps))

def compute_factorvae_score(model, dataset_imgs, dataset_classes, device, num_train=800, num_eval=800, batch_size=64):
    """
    Computes FactorVAE metric score (Kim & Mnih, 2018).
    """
    model.eval()
    active_factor_indices = [1, 2, 3, 4, 5]  # shape, scale, orientation, posX, posY
    num_factors = len(active_factor_indices)
    
    # Estimate global standard deviation for each latent dimension
    sample_size = min(5000, len(dataset_imgs))
    sample_indices = np.random.choice(len(dataset_imgs), size=sample_size, replace=False)
    with torch.no_grad():
        sample_x = torch.from_numpy(dataset_imgs[sample_indices]).float().unsqueeze(1).to(device)
        sample_mu, _ = model.encode(sample_x)
        global_std = torch.std(sample_mu, dim=0).cpu().numpy()
        
    active_dims = np.where(global_std > 0.05)[0]
    if len(active_dims) == 0:
        active_dims = np.arange(sample_mu.size(1))
    global_std[global_std < 1e-8] = 1.0

    def generate_batch_votes(num_votes):
        votes = []
        targets = []
        for _ in range(num_votes):
            k_idx = np.random.choice(num_factors)
            k = active_factor_indices[k_idx]
            val = np.random.choice(np.unique(dataset_classes[:, k]))
            matching = np.where(dataset_classes[:, k] == val)[0]
            
            replace = len(matching) < batch_size
            chosen = np.random.choice(matching, size=batch_size, replace=replace)
            
            with torch.no_grad():
                xb = torch.from_numpy(dataset_imgs[chosen]).float().unsqueeze(1).to(device)
                mu_b, _ = model.encode(xb)
                mu_b = mu_b.cpu().numpy()
                
            norm_var = np.var(mu_b / global_std, axis=0)
            dim_min_var = active_dims[np.argmin(norm_var[active_dims])]
            votes.append(dim_min_var)
            targets.append(k_idx)
            
        return np.array(votes), np.array(targets)

    train_votes, train_targets = generate_batch_votes(num_train)
    eval_votes, eval_targets = generate_batch_votes(num_eval)
    
    dim_to_factor = {}
    for d_idx in np.unique(train_votes):
        factor_votes = train_targets[train_votes == d_idx]
        dim_to_factor[d_idx] = np.bincount(factor_votes).argmax()
        
    predictions = np.array([dim_to_factor.get(d, 0) for d in eval_votes])
    accuracy = float(np.mean(predictions == eval_targets))
    return accuracy

def compute_dci(latents, factor_classes):
    """
    Computes DCI Disentanglement score (Eastwood & Williams, 2018; Locatello et al., 2019).
    latents: np.ndarray (N, d)
    factor_classes: np.ndarray (N, K)
    """
    N, d = latents.shape
    active_k = [k for k in range(factor_classes.shape[1]) if len(np.unique(factor_classes[:, k])) > 1]
    K = len(active_k)
    if K == 0:
        return 0.0
        
    importance_matrix = np.zeros((d, K))
    
    for idx, k in enumerate(active_k):
        y = factor_classes[:, k]
        num_classes = len(np.unique(y))
        if num_classes <= 5:
            clf = GradientBoostingClassifier(n_estimators=40, max_depth=3, random_state=42)
        else:
            clf = GradientBoostingRegressor(n_estimators=40, max_depth=3, random_state=42)
        clf.fit(latents, y)
        importance_matrix[:, idx] = clf.feature_importances_
        
    row_sums = importance_matrix.sum(axis=1)
    disentanglement_per_dim = np.zeros(d)
    
    for j in range(d):
        if row_sums[j] > 1e-12:
            p_jk = importance_matrix[j, :] / row_sums[j]
            nz = p_jk > 0
            entropy = -np.sum(p_jk[nz] * np.log(p_jk[nz])) / np.log(K) if K > 1 else 0.0
            disentanglement_per_dim[j] = 1.0 - entropy
        else:
            disentanglement_per_dim[j] = 0.0
            
    total_importance = row_sums.sum()
    if total_importance > 1e-12:
        weights = row_sums / total_importance
        return float(np.sum(weights * disentanglement_per_dim))
    return 0.0

@torch.no_grad()
def extract_eval_latents(model, eval_loader, device):
    """Extracts deterministic encoder means mu for the evaluation dataset."""
    model.eval()
    all_mu = []
    for x, _ in eval_loader:
        x = x.to(device)
        mu, _ = model.encode(x)
        all_mu.append(mu.cpu().numpy())
    return np.concatenate(all_mu, axis=0)



In [ ]:
def plot_latent_traversals(models_dict, sample_img, device, num_steps=7, val_range=(-3.0, 3.0), save_path=None):
    """
    Visualizes latent traversals across coordinates for given models.
    """
    if isinstance(sample_img, np.ndarray):
        sample_img = torch.from_numpy(sample_img).float()
    if sample_img.dim() == 2:
        sample_img = sample_img.unsqueeze(0).unsqueeze(0)
    elif sample_img.dim() == 3:
        sample_img = sample_img.unsqueeze(0)
    sample_img = sample_img.to(device)
    
    traversal_vals = np.linspace(val_range[0], val_range[1], num_steps)
    
    for model_name, model in models_dict.items():
        model.eval()
        with torch.no_grad():
            mu, _ = model.encode(sample_img)
            d = mu.size(1)
            
            fig, axes = plt.subplots(d, num_steps, figsize=(num_steps * 1.4, d * 1.4))
            fig.suptitle(f"Latent Traversals: {model_name}", fontsize=15, y=0.99)
            
            for j in range(d):
                for s, val in enumerate(traversal_vals):
                    z = mu.clone()
                    z[0, j] = val
                    recon = model.decode_probs(z)
                    img = recon[0, 0].cpu().numpy()
                    
                    ax = axes[j, s] if d > 1 else axes[s]
                    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
                    ax.axis("off")
                    if s == 0:
                        ax.text(-0.3, 0.5, f"z_{j}", transform=ax.transAxes,
                                verticalalignment="center", fontsize=11, fontweight="bold")
                        
            plt.tight_layout()
            if save_path:
                suffix = model_name.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "").lower()
                save_file = save_path if len(models_dict) == 1 else save_path.replace(".png", f"_{suffix}.png")
                plt.savefig(save_file, bbox_inches="tight", dpi=150)
                print(f"[Plot] Saved traversal figure: {save_file}")
            plt.show()
            plt.close()



In [ ]:
def train_betatcvae(train_loader, beta=6.0, epochs=15, lr=5e-4, device=device):
    print(f"\n==========================================")
    print(f"Starting Training: Beta-TCVAE (beta={beta}) with MWS")
    print(f"==========================================")
    model = ConvVAE(latent_dim=LATENT_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    N_train = len(train_loader.dataset)
    
    start_time = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        total_recon = 0.0
        total_kl = 0.0
        total_tc = 0.0
        total_loss = 0.0
        num_batches = 0
        
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            
            recon_logits, mu, logvar, z = model(x)
            
            # Reconstruction loss (Bernoulli negative log-likelihood)
            recon_loss = F.binary_cross_entropy_with_logits(recon_logits, x, reduction='none').sum(dim=(1, 2, 3)).mean()
            
            # Analytical Gaussian KL
            kl_loss = 0.5 * torch.sum(mu.pow(2) + logvar.exp() - 1.0 - logvar, dim=1).mean()
            
            # Minibatch Weighted Sampling (MWS) for Total Correlation
            tc, mi, dw_kl = compute_mws_tc(z, mu, logvar, dataset_size=N_train)
            
            # Beta-TCVAE objective: Recon + KL_analytic + (beta - 1) * TC
            loss = recon_loss + kl_loss + (beta - 1.0) * tc
            loss.backward()
            optimizer.step()
            
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
            total_tc += tc.item()
            total_loss += loss.item()
            num_batches += 1
            
        avg_recon = total_recon / num_batches
        avg_kl = total_kl / num_batches
        avg_tc = total_tc / num_batches
        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        
        if epoch % 2 == 0 or epoch == epochs or epoch == 1:
            print(f"Epoch [{epoch:02d}/{epochs:02d}] | Recon: {avg_recon:.2f} | KL: {avg_kl:.2f} | TC: {avg_tc:.2f} | Loss: {avg_loss:.2f} | Time: {elapsed:.1f}s")
            
    print(f"[Finished] Beta-TCVAE trained in {time.time() - start_time:.1f}s")
    return model



In [ ]:
model_betatcvae = train_betatcvae(
    train_loader=train_loader,
    beta=BETA,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    device=device
)



In [ ]:
print("\n==========================================")
print("Evaluating Beta-TCVAE Disentanglement Benchmarks")
print("==========================================")

# Extract evaluation latents
print("[1/4] Extracting validation representations...")
mu_tcvae = extract_eval_latents(model_betatcvae, eval_loader, device)

# 1. Reconstruction MSE
print("[2/4] Computing Reconstruction MSE...")
mse_tcvae = compute_reconstruction_mse(model_betatcvae, eval_loader, device)

# 2. Mutual Information Gap (MIG)
print("[3/4] Computing Mutual Information Gap (MIG)...")
mig_tcvae = compute_mig(mu_tcvae, eval_classes)

# 3. DCI Disentanglement
print("[4/4] Computing DCI Disentanglement and FactorVAE Scores...")
dci_tcvae = compute_dci(mu_tcvae, eval_classes)

# 4. FactorVAE Score
fvae_tcvae = compute_factorvae_score(model_betatcvae, all_imgs, all_latents_classes, device, num_train=600, num_eval=600)

# Build results DataFrame
results = [
    {
        "Model": "Beta-TCVAE",
        "Beta": BETA,
        "Reconstruction_MSE": round(mse_tcvae, 5),
        "MIG": round(mig_tcvae, 4),
        "DCI_Disentanglement": round(dci_tcvae, 4),
        "FactorVAE_Score": round(fvae_tcvae, 4)
    }
]

df_results = pd.DataFrame(results)
print("\n==========================================")
print("BETA-TCVAE RESULTS SUMMARY:")
print("==========================================")
print(df_results.to_string(index=False))

# Save results to CSV
csv_save_path = "results_betatcvae.csv"
df_results.to_csv(csv_save_path, index=False)
print(f"\n[Saved] Quantitative results saved to: {csv_save_path}")



In [ ]:
# Pick representative test image
test_sample = eval_imgs[20]

plot_latent_traversals(
    models_dict={"Beta-TCVAE (beta=6.0)": model_betatcvae},
    sample_img=test_sample,
    device=device,
    num_steps=7,
    val_range=(-3.0, 3.0),
    save_path="traversals_betatcvae.png"
)

